In [1]:
# %pip install -q kagglehub

In [2]:
# import os
# from dotenv import load_dotenv

# load_dotenv()

# api_key = os.getenv("API_KEY") 

# # link: https://www.kaggle.com/datasets/alessandrolobello/the-ultimate-earthquake-dataset-from-1990-2023

In [3]:
import os
import numpy as np
import pandas as pd

In [4]:
train_df = pd.read_csv(r"C:\Users\AYO_AYO\Desktop\Machine-Learning-Bootcamp\CSV-Excel\Datasets\titanic_dataset.csv", index_col=False)

train_df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


### Sample Test Run

In [5]:
print(len(train_df))

891


In [6]:
# ID like column
train_df["passenger_id"] = np.arange(1, len(train_df) + 1)

# Costant column
# Should be dropped before training.
train_df["constant_col"] = 1

# Transform all column to TitleCase
train_df.columns = train_df.columns.str.title()
# (train_df.columns.str.title(), inplace)

# Create a dirty version of Embark COlumn
train_df["Embark_Dirty"] = train_df["Embark_Town"].copy()

# Select 6 random non null columns
sample_idx = train_df["Embark_Dirty"].dropna().sample(6, random_state=11).index

# Apply simple dirty transformation
train_df.loc[sample_idx[:2], "Embark_Dirty"] = train_df.loc[sample_idx[:2],"Embark_Dirty"].str.upper()
train_df.loc[sample_idx[2:4], "Embark_Dirty"] = train_df.loc[sample_idx[2:4],"Embark_Dirty"].str.strip().apply(lambda x: f" {x} ")
train_df.loc[sample_idx[4:], "Embark_Dirty"] = "uknown"

### Data Quality Checklist
- Basic dataset overview (df.head(), df.info, df.describe)
- Missing values summary (df.isnull().sum())
- Drop duplicates (df.drop_duplicates())
- Data type validate (df["name"].dtype)
- Constant & Quasi constant columns
- ID like column == drop it
- String inconsistencies (clean)
- High nulls columns (fillna)
- High zero columns (numeric features)

**Basic Datasets Overview**

In [7]:
train_df.head()

,Survived,Pclass,Sex,Age,Sibsp,Parch,Fare,Embarked,Class,Who,Adult_Male,Deck,Embark_Town,Alive,Alone,Passenger_Id,Constant_Col,Embark_Dirty
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,1,1,Southampton
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,2,1,Cherbourg
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,3,1,Southampton
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,4,1,Southampton
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,5,1,Southampton


In [8]:
train_df.shape

(891, 18)

In [9]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 18 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Survived      891 non-null    int64  
 1   Pclass        891 non-null    int64  
 2   Sex           891 non-null    str    
 3   Age           714 non-null    float64
 4   Sibsp         891 non-null    int64  
 5   Parch         891 non-null    int64  
 6   Fare          891 non-null    float64
 7   Embarked      889 non-null    str    
 8   Class         891 non-null    str    
 9   Who           891 non-null    str    
 10  Adult_Male    891 non-null    bool   
 11  Deck          203 non-null    str    
 12  Embark_Town   889 non-null    str    
 13  Alive         891 non-null    str    
 14  Alone         891 non-null    bool   
 15  Passenger_Id  891 non-null    int64  
 16  Constant_Col  891 non-null    int64  
 17  Embark_Dirty  889 non-null    str    
dtypes: bool(2), float64(2), int64(6), str(8)


In [10]:
train_df.nunique()

Survived          2
Pclass            3
Sex               2
Age              88
Sibsp             7
Parch             7
Fare            248
Embarked          3
Class             3
Who               3
Adult_Male        2
Deck              7
Embark_Town       3
Alive             2
Alone             2
Passenger_Id    891
Constant_Col      1
Embark_Dirty      6
dtype: int64

**Missing Values Summary**

In [11]:
missing_count = train_df.isna().sum().sort_values(ascending = False)
missing_percent = (train_df.isna().mean() * 100).sort_values(ascending = False)

print(np.shape(train_df.isna().sum()))

missing_summary = pd.DataFrame(
    {
        "missing_count": missing_count,
        "missing_percent": round(missing_percent, 2)
    }
)

missing_summary

(18,)


,missing_count,missing_percent
Deck,688,77.22
Age,177,19.87
Embark_Dirty,2,0.22
Embarked,2,0.22
Embark_Town,2,0.22
Survived,0,0.00
Pclass,0,0.00
Parch,0,0.00
Sibsp,0,0.00
Sex,0,0.00


**Duplicates**

In [12]:
duplicates_mask = train_df.duplicated()
num_duplicated = duplicates_mask.sum()

print(f"Number of duplicate row {num_duplicated}")

# To remove duplicates
# remove_duplicate = train_df.drop_duplicates()

Number of duplicate row 0


**Data Type Validation**

In [13]:
train_df.columns

Index(['Survived', 'Pclass', 'Sex', 'Age', 'Sibsp', 'Parch', 'Fare',
       'Embarked', 'Class', 'Who', 'Adult_Male', 'Deck', 'Embark_Town',
       'Alive', 'Alone', 'Passenger_Id', 'Constant_Col', 'Embark_Dirty'],
      dtype='str')

In [14]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 18 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Survived      891 non-null    int64  
 1   Pclass        891 non-null    int64  
 2   Sex           891 non-null    str    
 3   Age           714 non-null    float64
 4   Sibsp         891 non-null    int64  
 5   Parch         891 non-null    int64  
 6   Fare          891 non-null    float64
 7   Embarked      889 non-null    str    
 8   Class         891 non-null    str    
 9   Who           891 non-null    str    
 10  Adult_Male    891 non-null    bool   
 11  Deck          203 non-null    str    
 12  Embark_Town   889 non-null    str    
 13  Alive         891 non-null    str    
 14  Alone         891 non-null    bool   
 15  Passenger_Id  891 non-null    int64  
 16  Constant_Col  891 non-null    int64  
 17  Embark_Dirty  889 non-null    str    
dtypes: bool(2), float64(2), int64(6), str(8)


In [15]:
expected_values = {
    "Survived": "int64",
    "Pclass": "int64",
    "Sex": "str",
    "Age": "float64",
    "Sibsp": "int64",
    "Parch": "int64",
    "Fare": "float64",
    "Embarked": "str",
    "Class": "str",
    "Who": "str",
    "Adult_Male": "bool",
    "Deck": "str",
    "Embark_Town": "str",
    "Alive": "str",
    "Alone": "bool",
    "Passenger_Id": "int64",
    "Constant_Col": "int64",
    "Embark_Dirty": "str"
}

for col, expected in expected_values.items():
    if col in train_df.columns:
        actual = train_df[col].dtype
        print(f"Actual: {actual}, Expected: {expected}")

Actual: int64, Expected: int64
Actual: int64, Expected: int64
Actual: str, Expected: str
Actual: float64, Expected: float64
Actual: int64, Expected: int64
Actual: int64, Expected: int64
Actual: float64, Expected: float64
Actual: str, Expected: str
Actual: str, Expected: str
Actual: str, Expected: str
Actual: bool, Expected: bool
Actual: str, Expected: str
Actual: str, Expected: str
Actual: str, Expected: str
Actual: bool, Expected: bool
Actual: int64, Expected: int64
Actual: int64, Expected: int64
Actual: str, Expected: str


In [16]:
# Data type conversion.

train_df["Fare"] = train_df["Fare"].astype("int64")
train_df["Fare"].dtype

dtype('int64')

**Constant & Quasi Value**

In [17]:
# Finding constant columns

n_row = len(train_df)
nunique = train_df.nunique()

constant_col = nunique[nunique == 1].index.to_list()
print(f"Constant columns: {constant_col}")

Constant columns: ['Constant_Col']


In [18]:
# Quasi Constant Column
# “Find columns where one value dominates more than 95% of the rows.”

quasi_constant_cols = []

for col in train_df.columns:
    top_freq = train_df[col].value_counts(normalize=True, dropna=False).values[0]
    if top_freq > .95 and col not in constant_col:
        quasi_constant_cols.append(col)
print(f"Quasi constant columns: {quasi_constant_cols}")

Quasi constant columns: []


**ID Like Columns**

In [19]:
n_row = len(train_df)
id_like_cols = []

for col in train_df.columns:
    if train_df[col].nunique(dropna=False) == n_row:
        id_like_cols.append(col)
print(id_like_cols)

n_row = len(train_df)
id_like_cols = []

id_like_cols.extend(col for col in train_df.columns if train_df[col].nunique() == n_row)
print(id_like_cols)

['Passenger_Id']
['Passenger_Id']


In [20]:
train_df['Passenger_Id'].value_counts(dropna=False).head()

Passenger_Id
1    1
2    1
3    1
4    1
5    1
Name: count, dtype: int64

**String Inconsistencies**

In [21]:
object_cols = train_df.select_dtypes(include=["str", "category"]).columns.to_list()
print(object_cols)

# Simple clean: strip spaces, convert the text to lower case

train_df["Embark_Clean"] = train_df["Embark_Dirty"].astype(str).str.lower().str.strip().replace("unknown", np.nan)

['Sex', 'Embarked', 'Class', 'Who', 'Deck', 'Embark_Town', 'Alive', 'Embark_Dirty']


**High Null Columns**

In [22]:
null_threshold = .4

high_null_threhold = missing_summary[missing_summary["missing_percent"] >= null_threshold * 100]

**High Zero Columns**

In [ ]:
# --------------------------------
numeric_cols = train_df.select_dtypes(include=[np.number]).columns.to_list()
zero_share = {col: (train_df[col] == 1).mean() for col in numeric_cols}
zero_share_series = pd.Series(zero_share).sort_values(ascending=False)
print(zero_share_series)

high_zero_threshold = .8
high_zero_cols = zero_share_series[zero_share_series >= high_zero_threshold]
print(high_zero_cols)

Constant_Col    1.000000
Survived        0.383838
Pclass          0.242424
Sibsp           0.234568
Parch           0.132435
Age             0.007856
Passenger_Id    0.001122
Fare            0.000000
dtype: float64
Constant_Col    1.0
dtype: float64


In [63]:
numeric_cols = train_df.select_dtypes(include=[np.number]).columns.to_list()
zero_share = {}

for col in numeric_cols:
    zero_share[col] = (train_df[col] == 0).mean()

print(pd.Series(zero_share.sort_values))

AttributeError: 'dict' object has no attribute 'sort_values'

**High Numeric Columns**

In [32]:
num_cols = train_df.select_dtypes(include=[np.number])
num_cols

,Survived,Pclass,Age,Sibsp,Parch,Fare,Passenger_Id,Constant_Col
0,0,3,22.0,1,0,7,1,1
1,1,1,38.0,1,0,71,2,1
2,1,3,26.0,0,0,7,3,1
3,1,1,35.0,1,0,53,4,1
4,0,3,35.0,0,0,8,5,1
...,...,...,...,...,...,...,...,...
886,0,2,27.0,0,0,13,887,1
887,1,1,19.0,0,0,30,888,1
888,0,3,NaN,1,2,23,889,1
889,1,1,26.0,0,0,30,890,1
